# 05: Hybrid Retrieval

Phase 2. Adds graph expansion to the dense+sparse retriever via Reciprocal Rank Fusion (RRF, k=60). Smoke-tests `hybrid_retrieve()`. Production implementation in `scripts/retriever.py`.

Run after notebook 04 (`data/cross_refs.jsonl` must exist).

## Cell group 1: Imports and constants

In [1]:
import re
import sys
import json
import time
from pathlib import Path
from collections import defaultdict

SCRIPTS_DIR = Path("scripts").resolve()
if str(SCRIPTS_DIR) not in sys.path:
    sys.path.insert(0, str(SCRIPTS_DIR))

from graph_build import load_clauses, load_cross_refs, build_graph, neighbours
from rank_bm25 import BM25Okapi
from langchain_chroma import Chroma
from langchain_huggingface import HuggingFaceEmbeddings

DATA_DIR    = Path("data")
CHROMA_DIR  = DATA_DIR / "chroma_db"
EMBED_MODEL = "all-MiniLM-L6-v2"
TOP_K       = 10   # clauses returned by hybrid_retrieve
RRF_K       = 60   # RRF smoothing constant (Cormack et al. 2009)
GRAPH_HOPS  = 2    # set by hop-count experiment in notebook 02
GRAPH_SEEDS = 5    # top-N from each retriever used as graph expansion seeds

print(f"DATA_DIR   : {DATA_DIR.resolve()}")
print(f"CHROMA_DIR : {CHROMA_DIR.resolve()}")
print(f"TOP_K={TOP_K}  RRF_K={RRF_K}  GRAPH_HOPS={GRAPH_HOPS}  GRAPH_SEEDS={GRAPH_SEEDS}")

DATA_DIR   : C:\Users\tsono\Documents\uoe\disertation\plan\implementation\data
CHROMA_DIR : C:\Users\tsono\Documents\uoe\disertation\plan\implementation\data\chroma_db
TOP_K=10  RRF_K=60  GRAPH_HOPS=2  GRAPH_SEEDS=5


In [2]:
from log_setup import setup_logging

logger = setup_logging("05_hybrid_retrieval")

00:50:58  INFO      === Notebook 05_hybrid_retrieval started — log: 2026-07-01_00-50-57_05_hybrid_retrieval.log ===


## Cell group 2: Load retrievers

Loads all three indices. ChromaDB reloads from disk (~1-2s). BM25 and the NetworkX graph are rebuilt from their source JSONL files.

In [3]:
STOPWORDS = {
    "the", "a", "an", "and", "or", "of", "in", "to", "for", "is", "are", "be",
    "was", "were", "will", "would", "shall", "should", "may", "might", "must",
    "not", "no", "by", "on", "at", "as", "from", "with", "this", "that", "it",
    "its", "such", "which", "who", "whom", "where", "when", "any", "all", "been",
    "has", "have", "had", "do", "does", "did", "if", "then", "than", "so", "also",
    "their", "they", "them", "these", "those", "but", "can", "into", "out", "up",
}

def tokenise(text: str) -> list[str]:
    return [
        t for t in re.findall(r"[a-z][a-z']*", text.lower())
        if t not in STOPWORDS and len(t) > 2
    ]

# --- Dense ---
t0 = time.time()
embeddings = HuggingFaceEmbeddings(model_name=EMBED_MODEL, model_kwargs={"device": "cpu"})
vectorstore = Chroma(
    persist_directory=str(CHROMA_DIR),
    embedding_function=embeddings,
    collection_name="aml_clauses",
)
print(f"ChromaDB loaded  : {vectorstore._collection.count()} docs  ({time.time()-t0:.1f}s)")

# --- Sparse ---
t0 = time.time()
clauses = load_clauses(DATA_DIR / "clauses.jsonl")
corpus_tokens = [tokenise(c["text"]) for c in clauses]
bm25 = BM25Okapi(corpus_tokens)
print(f"BM25 built       : {len(clauses)} clauses, vocab {len(bm25.idf)} terms  ({time.time()-t0:.2f}s)")

# --- Graph ---
t0 = time.time()
cross_refs = load_cross_refs(DATA_DIR / "cross_refs.jsonl")
G = build_graph(clauses, cross_refs=cross_refs)
clause_lookup = {c["clause_id"]: c for c in clauses}
print(f"NetworkX loaded  : {G.number_of_nodes()} nodes, {G.number_of_edges()} edges  ({time.time()-t0:.2f}s)")

00:51:12  INFO      NumExpr defaulting to 8 threads.
00:51:24  INFO      HTTP Request: HEAD https://huggingface.co/sentence-transformers/all-MiniLM-L6-v2/resolve/main/modules.json "HTTP/1.1 307 Temporary Redirect"
00:51:24  INFO      HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/sentence-transformers/all-MiniLM-L6-v2/1110a243fdf4706b3f48f1d95db1a4f5529b4d41/modules.json "HTTP/1.1 200 OK"
00:51:24  INFO      HTTP Request: HEAD https://huggingface.co/sentence-transformers/all-MiniLM-L6-v2/resolve/main/config_sentence_transformers.json "HTTP/1.1 307 Temporary Redirect"
00:51:24  INFO      HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/sentence-transformers/all-MiniLM-L6-v2/1110a243fdf4706b3f48f1d95db1a4f5529b4d41/config_sentence_transformers.json "HTTP/1.1 200 OK"
00:51:24  INFO      Loading SentenceTransformer model from sentence-transformers/all-MiniLM-L6-v2.
00:51:24  INFO      HTTP Request: HEAD https://huggingface.co/sentence-transformers/all-

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

00:51:25  INFO      HTTP Request: HEAD https://huggingface.co/sentence-transformers/all-MiniLM-L6-v2/resolve/main/processor_config.json "HTTP/1.1 404 Not Found"
00:51:26  INFO      HTTP Request: HEAD https://huggingface.co/sentence-transformers/all-MiniLM-L6-v2/resolve/main/preprocessor_config.json "HTTP/1.1 404 Not Found"
00:51:26  INFO      HTTP Request: HEAD https://huggingface.co/sentence-transformers/all-MiniLM-L6-v2/resolve/main/video_preprocessor_config.json "HTTP/1.1 404 Not Found"
00:51:26  INFO      HTTP Request: HEAD https://huggingface.co/sentence-transformers/all-MiniLM-L6-v2/resolve/main/preprocessor_config.json "HTTP/1.1 404 Not Found"
00:51:26  INFO      HTTP Request: HEAD https://huggingface.co/sentence-transformers/all-MiniLM-L6-v2/resolve/main/tokenizer_config.json "HTTP/1.1 307 Temporary Redirect"
00:51:26  INFO      HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/sentence-transformers/all-MiniLM-L6-v2/1110a243fdf4706b3f48f1d95db1a4f5529b4d41/toke

## Cell group 3: Individual retriever functions

Each function returns a list of dicts with `clause_id`, `source`, `text`, and `score`, sorted best-first.

In [4]:
def dense_retrieve(query: str, k: int = TOP_K) -> list[dict]:
    results = vectorstore.similarity_search_with_score(query, k=k)
    return [
        {
            "clause_id": doc.metadata["clause_id"],
            "source":    doc.metadata["source"],
            "text":      doc.page_content,
            "score":     float(score),
        }
        for doc, score in results
    ]


def sparse_retrieve(query: str, k: int = TOP_K) -> list[dict]:
    tokens = tokenise(query)
    scores = bm25.get_scores(tokens)
    top_idx = sorted(range(len(scores)), key=lambda i: scores[i], reverse=True)[:k]
    return [
        {
            "clause_id": clauses[i]["clause_id"],
            "source":    clauses[i]["source"],
            "text":      clauses[i]["text"],
            "score":     float(scores[i]),
        }
        for i in top_idx
    ]


def graph_expand(seed_ids: list[str], hops: int = GRAPH_HOPS) -> list[str]:
    """Return clause_ids reachable from seeds via CROSS_REFERENCES within `hops` steps.

    Ordered hop-1 before hop-2. Seeds excluded from output.
    """
    seen = set(seed_ids)
    ordered: list[str] = []
    frontier = set(seed_ids)

    for _ in range(hops):
        next_frontier: set[str] = set()
        for node in frontier:
            for nid in neighbours(G, node, rel="CROSS_REFERENCES", hops=1):
                if nid not in seen:
                    next_frontier.add(nid)
                    seen.add(nid)
                    ordered.append(nid)
        frontier = next_frontier

    return ordered


# Quick sanity check
_d = dense_retrieve("customer due diligence", k=3)
_s = sparse_retrieve("customer due diligence", k=3)
_g = graph_expand([_d[0]["clause_id"]], hops=1)

print("dense top-3 :", [r["clause_id"] for r in _d])
print("sparse top-3:", [r["clause_id"] for r in _s])
print("graph (1-hop from dense[0]):", _g[:5])

dense top-3 : ['jmlsg_1_5.1', 'jmlsg_2_14.24', 'jmlsg_2_9.16']
sparse top-3: ['mlr_2017_sch6_para7', 'jmlsg_2_14.24', 'jmlsg_2_18.93']
graph (1-hop from dense[0]): []


## Cell group 4: RRF fusion

`rrf_fuse` takes any number of ranked lists. `hybrid_retrieve` wires the three retrievers together: dense and sparse run independently, then graph expansion seeds from their combined top-N.

In [5]:
def rrf_fuse(ranked_lists: list[list[dict]], k: int = RRF_K) -> list[dict]:
    """Reciprocal Rank Fusion (Cormack et al. 2009).

    score(d) = Σ 1 / (k + rank(d))  for each retriever list that contains d
    k=60 is the default from the original paper, not tuned on this corpus.
    """
    scores: dict[str, float] = {}
    for ranked in ranked_lists:
        for rank, item in enumerate(ranked):
            cid = item["clause_id"]
            scores[cid] = scores.get(cid, 0.0) + 1.0 / (k + rank + 1)

    return [
        {"clause_id": cid, "rrf_score": score}
        for cid, score in sorted(scores.items(), key=lambda x: x[1], reverse=True)
    ]


def hybrid_retrieve(query: str, k: int = TOP_K) -> list[dict]:
    """Dense + sparse + graph expansion → RRF → top-k clauses.

    Returns list[dict] with: clause_id, source, text, rrf_score, rank
    """
    dense_results  = dense_retrieve(query, k=k)
    sparse_results = sparse_retrieve(query, k=k)

    seeds = list(
        {r["clause_id"] for r in dense_results[:GRAPH_SEEDS]} |
        {r["clause_id"] for r in sparse_results[:GRAPH_SEEDS]}
    )
    graph_ids = graph_expand(seeds, hops=GRAPH_HOPS)
    graph_results = [
        {
            "clause_id": cid,
            "source":    clause_lookup[cid]["source"],
            "text":      clause_lookup[cid]["text"],
        }
        for cid in graph_ids
        if cid in clause_lookup
    ]

    fused = rrf_fuse([dense_results, sparse_results, graph_results])

    out = []
    for rank, item in enumerate(fused[:k]):
        cid = item["clause_id"]
        clause = clause_lookup.get(cid, {})
        out.append({
            "clause_id": cid,
            "source":    clause.get("source", ""),
            "text":      clause.get("text", ""),
            "rrf_score": round(item["rrf_score"], 6),
            "rank":      rank + 1,
        })

    return out


# Quick sanity check
_h = hybrid_retrieve("customer due diligence", k=5)
for r in _h:
    print(f"  #{r['rank']}  {r['clause_id']:<40}  rrf={r['rrf_score']:.5f}")

  #1  jmlsg_2_14.24                             rrf=0.03226
  #2  jmlsg_1_5.1                               rrf=0.01639
  #3  mlr_2017_sch6_para7                       rrf=0.01639
  #4  mlr_2017_reg_28                           rrf=0.01639
  #5  mlr_2017_reg_34                           rrf=0.01613


## Cell group 5: Smoke test

Five AML queries covering different retrieval challenges: exact-anchor, statutory offence, EDD/PEPs, SAR obligations, and defined terms.

In [6]:
# Each query accepts multiple valid clause_ids: any one in top-5 counts as a hit.
# JMLSG guidance clauses are semantically valid alongside the primary statutory clauses
# because they have more detailed text on the same topics.
SMOKE_QUERIES = [
    (
        "customer due diligence requirements",
        {"mlr_2017_reg_28", "mlr_2017_reg_27", "jmlsg_2_14.24", "jmlsg_1_5.1"},
    ),
    (
        "concealment of criminal property",
        {"poca_2002_s327"},
    ),
    (
        "politically exposed persons enhanced due diligence",
        {"mlr_2017_reg_35", "mlr_2017_reg_37", "mlr_2017_reg_38", "fatf_40_R12", "jmlsg_2_7.6"},
    ),
    (
        "suspicious activity report obligation",
        {"poca_2002_s330", "jmlsg_2_16.38", "jmlsg_1_6.86", "mlr_2017_reg_104"},
    ),
    (
        "beneficial owner definition",
        {"mlr_2017_reg_6", "jmlsg_1_5.3.149", "jmlsg_2_7.46"},
    ),
]

hits = 0
for query, expected_set in SMOKE_QUERIES:
    results = hybrid_retrieve(query, k=5)
    top_ids = [r["clause_id"] for r in results]
    hit = bool(expected_set & set(top_ids))
    hits += hit
    status = "HIT " if hit else "MISS"
    matched = expected_set & set(top_ids)
    print(f"[{status}] {query!r}")
    print(f"       matched : {matched or 'none'}")
    print(f"       top-5   : {top_ids}")
    print()

print(f"Smoke test: {hits}/{len(SMOKE_QUERIES)} queries hit an expected clause in top-5")
assert hits >= 4, f"Expected ≥4/5 hits, got {hits}"
print("Assertion passed.")

[HIT ] 'customer due diligence requirements'
       matched : {'jmlsg_2_14.24'}
       top-5   : ['jmlsg_2_14.24', 'mlr_2017_sch6_para7', 'fatf_40_R10', 'jmlsg_2_9.16', 'jmlsg_2_13.62']

[HIT ] 'concealment of criminal property'
       matched : {'poca_2002_s327'}
       top-5   : ['poca_2002_s327', 'jmlsg_2_14.3', 'poca_2002_s338', 'poca_2002_s82', 'poca_2002_s342']

[HIT ] 'politically exposed persons enhanced due diligence'
       matched : {'jmlsg_2_7.6', 'mlr_2017_reg_37', 'fatf_40_R12'}
       top-5   : ['fatf_40_R12', 'jmlsg_2_7.6', 'mlr_2017_sch6_para7', 'mlr_2017_reg_34', 'mlr_2017_reg_37']

[HIT ] 'suspicious activity report obligation'
       matched : {'jmlsg_1_6.86', 'jmlsg_2_16.38', 'mlr_2017_reg_104'}
       top-5   : ['jmlsg_2_16.38', 'jmlsg_1_6.86', 'jmlsg_2_22.22', 'jmlsg_1_6.12', 'mlr_2017_reg_104']

[HIT ] 'beneficial owner definition'
       matched : {'jmlsg_2_7.46', 'jmlsg_1_5.3.149'}
       top-5   : ['jmlsg_1_5.3.149', 'jmlsg_2_7.46', 'jmlsg_1_5.3.265', 'fca_fc

## Cell group 6: Retriever comparison

Side-by-side comparison of hybrid vs dense-only vs sparse-only for each smoke test query. Shows where the hybrid result differs from either individual retriever.

In [7]:
def dense_only(query: str, k: int = TOP_K) -> list[str]:
    return [r["clause_id"] for r in dense_retrieve(query, k=k)]

def sparse_only(query: str, k: int = TOP_K) -> list[str]:
    return [r["clause_id"] for r in sparse_retrieve(query, k=k)]

def hybrid_only(query: str, k: int = TOP_K) -> list[str]:
    return [r["clause_id"] for r in hybrid_retrieve(query, k=k)]

COMPARE_K = 5

print(f"{'Query':<48}  {'Method':<8}  Top-{COMPARE_K} clause_ids")
print("-" * 120)

for query, expected in SMOKE_QUERIES:
    d = dense_only(query, k=COMPARE_K)
    s = sparse_only(query, k=COMPARE_K)
    h = hybrid_only(query, k=COMPARE_K)

    for method, ids in [("dense", d), ("sparse", s), ("hybrid", h)]:
        marker = "* " if expected in ids else "  "
        label = query[:46] + ".." if len(query) > 46 else query
        q_col = f"{label:<48}" if method == "dense" else " " * 48
        print(f"{q_col}  {method:<8}  {marker}{ids}")
    print()

Query                                             Method    Top-5 clause_ids
------------------------------------------------------------------------------------------------------------------------
customer due diligence requirements               dense       ['jmlsg_2_14.24', 'jmlsg_2_9.16', 'jmlsg_1_5.1', 'jmlsg_2_16.16', 'jmlsg_1_5.1.13']
                                                  sparse      ['mlr_2017_sch6_para7', 'jmlsg_2_13.62', 'jmlsg_2_16.17', 'fatf_40_INR22', 'jmlsg_2_7.33']
                                                  hybrid      ['jmlsg_2_14.24', 'mlr_2017_sch6_para7', 'fatf_40_R10', 'jmlsg_2_9.16', 'jmlsg_2_13.62']

concealment of criminal property                  dense       ['poca_2002_s327', 'poca_2002_s82', 'poca_2002_s148', 'poca_2002_s230', 'poca_2002_s127M']
                                                  sparse      ['jmlsg_2_14.3', 'poca_2002_s342', 'poca_2002_s327', 'poca_2002_s329', 'poca_2002_s328']
                                               

## Cell group 7: Verify scripts/retriever.py

Imports the production module and checks that it returns the same top-1 result as the inline implementation. This catches any divergence introduced during the export.

In [8]:
import importlib
import retriever as ret_module
importlib.reload(ret_module)  # pick up latest file state

vs2, bm2, clauses2, G2 = ret_module.load_retrievers(DATA_DIR, CHROMA_DIR)

TEST_QUERY = "customer due diligence requirements"
inline_top1  = hybrid_retrieve(TEST_QUERY, k=5)[0]["clause_id"]
module_top1  = ret_module.hybrid_retrieve(
    TEST_QUERY, vs2, bm2, clauses2, G2,
    k=5, graph_hops=GRAPH_HOPS, rrf_k=RRF_K
)[0]["clause_id"]

print(f"Inline  top-1: {inline_top1}")
print(f"Module  top-1: {module_top1}")
assert inline_top1 == module_top1, (
    f"Mismatch: inline={inline_top1!r}  module={module_top1!r}"
)
print("scripts/retriever.py matches inline implementation.")

00:51:31  INFO      Loading all retrievers (dense+sparse+graph) from data
00:51:31  INFO      Loading dense+sparse retrievers from data
00:51:31  INFO      HTTP Request: HEAD https://huggingface.co/sentence-transformers/all-MiniLM-L6-v2/resolve/main/modules.json "HTTP/1.1 307 Temporary Redirect"
00:51:31  INFO      HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/sentence-transformers/all-MiniLM-L6-v2/1110a243fdf4706b3f48f1d95db1a4f5529b4d41/modules.json "HTTP/1.1 200 OK"
00:51:31  INFO      HTTP Request: HEAD https://huggingface.co/sentence-transformers/all-MiniLM-L6-v2/resolve/main/config_sentence_transformers.json "HTTP/1.1 307 Temporary Redirect"
00:51:31  INFO      HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/sentence-transformers/all-MiniLM-L6-v2/1110a243fdf4706b3f48f1d95db1a4f5529b4d41/config_sentence_transformers.json "HTTP/1.1 200 OK"
00:51:31  INFO      Loading SentenceTransformer model from sentence-transformers/all-MiniLM-L6-v2.
00:51

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

00:51:32  INFO      HTTP Request: HEAD https://huggingface.co/sentence-transformers/all-MiniLM-L6-v2/resolve/main/processor_config.json "HTTP/1.1 404 Not Found"
00:51:32  INFO      HTTP Request: HEAD https://huggingface.co/sentence-transformers/all-MiniLM-L6-v2/resolve/main/preprocessor_config.json "HTTP/1.1 404 Not Found"
00:51:32  INFO      HTTP Request: HEAD https://huggingface.co/sentence-transformers/all-MiniLM-L6-v2/resolve/main/video_preprocessor_config.json "HTTP/1.1 404 Not Found"
00:51:32  INFO      HTTP Request: HEAD https://huggingface.co/sentence-transformers/all-MiniLM-L6-v2/resolve/main/preprocessor_config.json "HTTP/1.1 404 Not Found"
00:51:33  INFO      HTTP Request: HEAD https://huggingface.co/sentence-transformers/all-MiniLM-L6-v2/resolve/main/tokenizer_config.json "HTTP/1.1 307 Temporary Redirect"
00:51:33  INFO      HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/sentence-transformers/all-MiniLM-L6-v2/1110a243fdf4706b3f48f1d95db1a4f5529b4d41/toke